# Stage D (Part 2) — Dynamic ERA5 **Daily** Features & Train Sets

**Purpose**
Engineer **daily** meteorological features from ERA5 at the basin level, join **static basin attributes**, and assemble **training datasets** for the 3 modeled basins.

---

## Inputs

* ERA5 daily (basin-level): `data/era5/era5_aligned/basin_level/era5_basin_daily.parquet`
* Targets daily: `data/modeling/targets/targets_discharge_basin_daily.parquet`
* Static attributes (from Part 1): `data/modeling/static/basin_attributes.parquet`
* Station→basin selections (sanity): `data/modeling/targets/meta/station_to_basin_name.csv` + `data/boundaries/processed/basin_lookup.csv`

---

## What this notebook does

1. **Resolve project root** and set paths/knobs (windows, API half-lives, ARX lags).
2. **Load & constrain** ERA5 and targets: keep only the **3 basins** and intersected **date window**; normalize to `date_local` (naive midnight).
3. **Variable catalog & base derivations**

   * Classify ERA5 vars as **flux-like** (e.g., precipitation, runoff, snowmelt, radiation, PET) vs **state-like** (e.g., temperature, dewpoint, soil T, snow depth).
   * Derive **wind\_speed** from U/V; derive **VPD** from temperature & dewpoint.
4. **Leak-safe feature engineering (daily)**

   * **Flux windows (sums):** 1, 3, 7, 14, 30 days → `*_sum_{w}d` (uses **past** days only).
   * **State windows (means):** 1, 3, 7, 14 days → `*_mean_{w}d` (past days only).
   * **API (Antecedent Precipitation Index):** exponential memory of precip with **3** and **7** day half-lives → `api_d3`, `api_d7` (shifted to avoid leakage).
   * **Calendar/condition flags:** `month`, `doy_sin`, `doy_cos`, `is_monsoon` (Jun–Sep), `is_freezing` (T ≤ 0 °C), `has_snowpack` (snow\_depth > 0).
5. **Join statics**: merge `basin_attributes.parquet` onto each `(basin_id, date_local)` row.
6. **Save features-only table** → `data/modeling/features/era5_features_basin_daily.parquet`.
7. **Build training datasets**

   * **Exogenous-only:** features ⊕ target (`discharge_cms`, `qc_any`) → `data/modeling/datasets/train_basin_daily_exogenous.parquet`.
   * **ARX (optional):** add discharge lags `q_lag_{1,2,3,7,14,30}d` and rolling stats (`q_roll7_mean`, `q_roll14_std`); drop rows without full lag history → `data/modeling/datasets/train_basin_daily_arx.parquet`.

---

## Assumptions & conventions

* **Cadence:** daily only (no 6-hour expansion).
* **Time zone:** Bhutan local; `date_local` normalized to naive midnight.
* **Leak safety:** all rolling/lag features use **shifted** history (no future info).
* **Units:** ERA5 kept in native units; if precip is meters/day you can later add a `_mm` copy for readability.
* **QC:** `qc_any` from targets is carried through; no filtering/imputation applied here.

---

## QA printed by the notebook

* Modeled `basin_id`s; row counts before/after subsetting.
* Date window used (start → end).
* Detected flux/state variables; confirmation of derived `wind_speed`/`vpd_kpa`.
* Feature NA diagnostics: `feature_na_count` / `feature_na_frac`.
* Train set sizes; ARX rows dropped due to lag history.

---

## Outputs

* **Features (daily):** `data/modeling/features/era5_features_basin_daily.parquet`
* **Train — exogenous:** `data/modeling/datasets/train_basin_daily_exogenous.parquet`
* **Train — ARX (optional):** `data/modeling/datasets/train_basin_daily_arx.parquet`

---

## Tunable knobs (set at top of notebook)

* Flux windows: `[1, 3, 7, 14, 30]`
* State windows: `[1, 3, 7, 14]`
* API half-lives: `[3, 7]` days
* ARX lags: `[1, 2, 3, 7, 14, 30]` and rolling stats: `(7, mean)`, `(14, std)`
* Optional `ERA5_KEEP` whitelist to slim variables before engineering.


In [1]:
# === Resolve Project Root ===
from pathlib import Path
import subprocess

def get_project_root(max_up=6):
    try:
        root = subprocess.check_output(["git","rev-parse","--show-toplevel"], text=True).strip()
        if root:
            return Path(root)
    except Exception:
        pass
    p = Path.cwd()
    for _ in range(max_up):
        if (p/"data").exists() and (p/"code").exists():
            return p
        if (p/".git").exists():
            return p
        p = p.parent
    return Path.cwd()

PROJECT_ROOT = get_project_root()
print("Project root:", PROJECT_ROOT)


Project root: /Users/liuq13/bhutan_climate_modeling


In [2]:
# === Paths & Settings ===
import pandas as pd
import numpy as np
from pathlib import Path

# Inputs
ERA5_DAILY_PARQUET     = PROJECT_ROOT / "data/era5/era5_aligned/basin_level/era5_basin_daily.parquet"
TARGETS_DAILY_PARQUET  = PROJECT_ROOT / "data/modeling/targets/targets_discharge_basin_daily.parquet"
STATIC_ATTR_PARQUET    = PROJECT_ROOT / "data/modeling/static/basin_attributes.parquet"
STATION_TO_BASIN_CSV   = PROJECT_ROOT / "data/modeling/targets/meta/station_to_basin_name.csv"
BASIN_LOOKUP_CSV       = PROJECT_ROOT / "data/boundaries/processed/basin_lookup.csv"

# Outputs
FEATURES_DAILY_PARQUET = PROJECT_ROOT / "data/modeling/features/era5_features_basin_daily.parquet"
TRAIN_EXOG_PARQUET     = PROJECT_ROOT / "data/modeling/datasets/train_basin_daily_exogenous.parquet"
TRAIN_ARX_PARQUET      = PROJECT_ROOT / "data/modeling/datasets/train_basin_daily_arx.parquet"

FEATURES_DAILY_PARQUET.parent.mkdir(parents=True, exist_ok=True)
TRAIN_EXOG_PARQUET.parent.mkdir(parents=True, exist_ok=True)

# Feature knobs (daily cadence)
FLUX_WINDOWS  = [1, 3, 7, 14, 30]      # sums of past N full days
STATE_WINDOWS = [1, 3, 7, 14]          # means of past N full days
API_HALFLIFE  = [3, 7]                 # days (for antecedent precip index)
ARX_LAGS      = [1, 2, 3, 7, 14, 30]   # discharge lags (days)
ROLL_Q_STATS  = [(7, "mean"), (14, "std")]  # discharge rolling stats
PRODUCE_ARX   = True                   # also build ARX dataset (exogenous + lagged Q)

# Optional whitelist to slim ERA5 before engineering (None = keep all)
ERA5_KEEP = None


## Load Mapping (3 modeled basins) & Data

In [14]:
# === Load mapping: 3 modeled basins ===
st2name = pd.read_csv(STATION_TO_BASIN_CSV)  # station_name, use_for_model, basin_name
lookup  = pd.read_csv(BASIN_LOOKUP_CSV)      # basin_id, basin_name, (optional slug)

mapping = (st2name.merge(lookup, on="basin_name", how="left")
                 .query("use_for_model == True")
                 .copy())
if mapping["basin_id"].isna().any():
    raise ValueError("Some chosen stations have basin_name not found in lookup:\n"
                     + str(mapping[mapping["basin_id"].isna()][["station_name","basin_name"]]))

MODELED_BASINS = sorted(mapping["basin_id"].unique().tolist())
print("Modeled basins:", MODELED_BASINS)

# === Load ERA5 daily (already aggregated) ===
e = pd.read_parquet(ERA5_DAILY_PARQUET)

# Ensure date_local as naive midnight Timestamp
if "date_local" in e.columns:
    e["date_local"] = pd.to_datetime(e["date_local"], errors="coerce").dt.tz_localize(None).dt.normalize()
elif "datetime_local" in e.columns:
    e["date_local"] = pd.to_datetime(e["datetime_local"], errors="coerce").dt.tz_localize(None).dt.normalize()
    e = e.drop(columns=["datetime_local"])
else:
    raise ValueError("ERA5 daily must have 'date_local' or 'datetime_local'.")

# Keep only the 3 modeled basins
e = e[e["basin_id"].isin(MODELED_BASINS)].copy()

# Optional prune to a variable subset
if ERA5_KEEP is not None:
    keep = ["basin_id","date_local"] + [c for c in ERA5_KEEP if c in e.columns]
    e = e[keep]

print("ERA5 daily shape after subset:", e.shape)

Modeled basins: [3.0, 6.0, 8.0]
ERA5 daily shape after subset: (51021, 17)


In [11]:
# === Load targets daily ===
y = pd.read_parquet(TARGETS_DAILY_PARQUET)
y["date_local"] = pd.to_datetime(y["date_local"]).dt.tz_localize(None).dt.normalize()
y = y[y["basin_id"].isin(MODELED_BASINS)].copy()
print("Targets daily shape:", y.shape)
y

Targets daily shape: (29840, 5)


,basin_id,basin_name,date_local,discharge_cms,qc_any
0,3.0,Mangdechhu,2000-01-01,14.783000,False
1,3.0,Mangdechhu,2000-01-02,14.620000,False
2,3.0,Mangdechhu,2000-01-03,14.565000,False
3,3.0,Mangdechhu,2000-01-04,14.031000,False
4,3.0,Mangdechhu,2000-01-05,14.032000,False
...,...,...,...,...,...
29835,8.0,Punatsangchhu,2023-12-27,74.138000,False
29836,8.0,Punatsangchhu,2023-12-28,74.247002,False
29837,8.0,Punatsangchhu,2023-12-29,74.027000,False
29838,8.0,Punatsangchhu,2023-12-30,73.889999,False


In [13]:
# === Intersect date window to ensure alignment ===
start = max(e["date_local"].min(), y["date_local"].min())
end   = min(e["date_local"].max(), y["date_local"].max())
e = e[(e["date_local"] >= start) & (e["date_local"] <= end)].copy()
y = y[(y["date_local"] >= start) & (y["date_local"] <= end)].copy()

print("Date window:", start.date(), "→", end.date())
y

Date window: 1991-04-22 → 2024-12-31


,basin_id,basin_name,date_local,discharge_cms,qc_any
0,3.0,Mangdechhu,2000-01-01,14.783000,False
1,3.0,Mangdechhu,2000-01-02,14.620000,False
2,3.0,Mangdechhu,2000-01-03,14.565000,False
3,3.0,Mangdechhu,2000-01-04,14.031000,False
4,3.0,Mangdechhu,2000-01-05,14.032000,False
...,...,...,...,...,...
29835,8.0,Punatsangchhu,2023-12-27,74.138000,False
29836,8.0,Punatsangchhu,2023-12-28,74.247002,False
29837,8.0,Punatsangchhu,2023-12-29,74.027000,False
29838,8.0,Punatsangchhu,2023-12-30,73.889999,False


## Variable Catalog & Base Derivations (wind, VPD)

In [4]:
# === Identify numeric ERA5 variables (exclude IDs/time) ===
exclude = {"basin_id","date_local"}
num_cols = [c for c in e.columns if c not in exclude and pd.api.types.is_numeric_dtype(e[c])]
print("Detected numeric ERA5 columns:", len(num_cols))

# Heuristic classification
def classify_flux_state(cols):
    flux_like, state_like = set(), set()
    for c in cols:
        lc = c.lower()
        if any(k in lc for k in ["precip", "runoff", "snowmelt", "evap", "radiation", "ssrd"]):
            flux_like.add(c)
        else:
            state_like.add(c)
    return sorted(flux_like), sorted(state_like)

flux_vars, state_vars = classify_flux_state(num_cols)

# Add wind_speed if possible (from U/V) and reclassify as state-like
def add_wind_speed(df, state_vars):
    # Try common names
    candidates = [
        ("u10", "v10"),  # ECMWF 10m wind
        ("u_component_of_wind_10m", "v_component_of_wind_10m"),
        ("wind_u", "wind_v"), ("u", "v"),
    ]
    for u_name, v_name in candidates:
        u = next((c for c in df.columns if u_name == c or u_name in c.lower()), None)
        v = next((c for c in df.columns if v_name == c or v_name in c.lower()), None)
        if u and v:
            df["wind_speed"] = np.sqrt(df[u]**2 + df[v]**2)
            return df, state_vars + (["wind_speed"] if "wind_speed" not in state_vars else [])
    # If a wind_speed already exists, keep it
    if "wind_speed" in df.columns and "wind_speed" not in state_vars:
        state_vars = state_vars + ["wind_speed"]
    return df, state_vars

e, state_vars = add_wind_speed(e, state_vars)

# Prepare temperature & dew point for VPD (auto-detect Kelvin vs Celsius)
def to_celsius(series: pd.Series) -> pd.Series:
    med = series.median(skipna=True)
    # If median > 200, assume Kelvin
    if pd.notna(med) and med > 200:
        return series - 273.15
    return series

def find_col(name_hints):
    for h in name_hints:
        c = next((c for c in e.columns if h == c or h in c.lower()), None)
        if c: return c
    return None

temp_col = find_col(["t2m","temperature","2m_temperature"])
dew_col  = find_col(["d2m","dewpoint","dew_point","2m_dewpoint"])

if temp_col and dew_col:
    T_c  = to_celsius(e[temp_col])
    Td_c = to_celsius(e[dew_col])
    # Magnus (kPa)
    es  = 0.6108 * np.exp(17.27 * T_c  / (T_c  + 237.3))
    ea  = 0.6108 * np.exp(17.27 * Td_c / (Td_c + 237.3))
    e["vpd_kpa"] = (es - ea).clip(lower=0)
    if "vpd_kpa" not in state_vars:
        state_vars.append("vpd_kpa")

print("Flux-like vars:", flux_vars[:12], "...")
print("State-like vars (incl. derived):", [v for v in state_vars if v not in flux_vars][:12], "...")


Detected numeric ERA5 columns: 15
Flux-like vars: ['potential_evaporation', 'precipitation', 'runoff', 'snowmelt', 'solar_radiation', 'sub_surface_runoff', 'surface_runoff'] ...
State-like vars (incl. derived): ['dewpoint', 'low_coverage', 'n_cells', 'snow_depth', 'soil_temperature', 'temperature', 'wind_u', 'wind_v', 'wind_speed', 'vpd_kpa'] ...


## Rolling & Lag Functions (Leak-safe)

In [5]:
# === Rolling & lag helpers (grouped by basin, using only past days) ===

def strict_sum_past(s: pd.Series, window: int) -> pd.Series:
    # sum over the previous N full days, not including today
    return s.shift(1).rolling(window=window, min_periods=window).sum()

def strict_mean_past(s: pd.Series, window: int) -> pd.Series:
    return s.shift(1).rolling(window=window, min_periods=window).mean()

def api_series(s: pd.Series, half_life_days: int) -> pd.Series:
    # Exponential memory on daily precip (or any flux), using only past plus today's input.
    k = 0.5 ** (1.0 / half_life_days)
    out = np.empty(len(s))
    out[:] = np.nan
    acc = 0.0
    for i, val in enumerate(s.fillna(0.0).values):
        acc = val + k * acc
        out[i] = acc
    # Shift by 1 day so API_t uses data up to t-1 (no leakage)
    return pd.Series(out, index=s.index).shift(1)


## Build Daily Features (flux/state windows, API, calendar)

In [6]:
# === Build engineered features per basin_id ===
e = e.sort_values(["basin_id","date_local"]).reset_index(drop=True)

# Work on a copy that we will extend
feat = e[["basin_id","date_local"]].copy()

# Flux windows
for v in flux_vars:
    for w in FLUX_WINDOWS:
        col = f"{v}_sum_{w}d"
        feat[col] = (e.groupby("basin_id")[v]
                       .apply(lambda s: strict_sum_past(s, w))
                       .reset_index(level=0, drop=True))

# State windows
for v in state_vars:
    for w in STATE_WINDOWS:
        col = f"{v}_mean_{w}d"
        feat[col] = (e.groupby("basin_id")[v]
                       .apply(lambda s: strict_mean_past(s, w))
                       .reset_index(level=0, drop=True))

# API from precip if a precip-like column exists
precip_col = next((c for c in e.columns if "precip" in c.lower()), None)
if precip_col:
    for h in API_HALFLIFE:
        col = f"api_d{h}"
        feat[col] = (e.groupby("basin_id")[precip_col]
                       .apply(lambda s: api_series(s, h))
                       .reset_index(level=0, drop=True))

# Calendar & simple condition flags (uses original ERA5 where needed)
feat["month"] = feat["date_local"].dt.month
doy = feat["date_local"].dt.dayofyear
feat["doy_sin"] = np.sin(2*np.pi * (doy/365.25))
feat["doy_cos"] = np.cos(2*np.pi * (doy/365.25))
feat["is_monsoon"] = feat["month"].between(6,9).astype(int)

# Optional freezing/snowpack flags if we have T and snow depth
snow_depth_col = next((c for c in e.columns if "snow_depth" in c.lower()), None)
if temp_col:
    T_c = to_celsius(e[temp_col])
    feat["is_freezing"] = (T_c <= 0).astype(int)
else:
    feat["is_freezing"] = 0

if snow_depth_col:
    feat["has_snowpack"] = (e[snow_depth_col] > 0).astype(int)
else:
    feat["has_snowpack"] = 0


In [15]:
feat

,basin_id,date_local,potential_evaporation_sum_1d,potential_evaporation_sum_3d,potential_evaporation_sum_7d,potential_evaporation_sum_14d,potential_evaporation_sum_30d,precipitation_sum_1d,precipitation_sum_3d,precipitation_sum_7d,...,slope_p90_deg,relief_m,log_acc_mean,log_acc_max,pct_slope_gt_a,pct_slope_gt_b,acc_mean,acc_max,feature_na_count,feature_na_frac
0,3.0,1991-04-22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,35.971821,6956.0,6.976817,13.794746,77.342031,25.226937,1070.502172,979449.0,77,0.726415
1,3.0,1991-04-23,-0.000256,NaN,NaN,NaN,NaN,0.001312,NaN,NaN,...,35.971821,6956.0,6.976817,13.794746,77.342031,25.226937,1070.502172,979449.0,58,0.547170
2,3.0,1991-04-24,-0.000298,NaN,NaN,NaN,NaN,0.002204,NaN,NaN,...,35.971821,6956.0,6.976817,13.794746,77.342031,25.226937,1070.502172,979449.0,58,0.547170
3,3.0,1991-04-25,-0.000290,-0.000845,NaN,NaN,NaN,0.002312,0.005829,NaN,...,35.971821,6956.0,6.976817,13.794746,77.342031,25.226937,1070.502172,979449.0,41,0.386792
4,3.0,1991-04-26,-0.000319,-0.000908,NaN,NaN,NaN,0.000747,0.005263,NaN,...,35.971821,6956.0,6.976817,13.794746,77.342031,25.226937,1070.502172,979449.0,41,0.386792
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36919,8.0,2024-12-27,-0.000212,-0.000729,-0.001557,-0.003308,-0.006572,0.000111,0.000580,0.002076,...,36.667324,6993.0,7.276396,14.060585,79.314210,26.663519,1444.768641,1277715.0,0,0.000000
36920,8.0,2024-12-28,-0.000317,-0.000761,-0.001675,-0.003462,-0.006674,0.000000,0.000412,0.001725,...,36.667324,6993.0,7.276396,14.060585,79.314210,26.663519,1444.768641,1277715.0,0,0.000000
36921,8.0,2024-12-29,-0.000310,-0.000839,-0.001778,-0.003584,-0.006738,0.000080,0.000191,0.001142,...,36.667324,6993.0,7.276396,14.060585,79.314210,26.663519,1444.768641,1277715.0,0,0.000000
36922,8.0,2024-12-30,-0.000225,-0.000851,-0.001777,-0.003488,-0.006637,0.000880,0.000960,0.001778,...,36.667324,6993.0,7.276396,14.060585,79.314210,26.663519,1444.768641,1277715.0,0,0.000000


## Join Static Basin Attributes

In [16]:
# === Join static attributes onto each (basin_id, date_local) row ===
static = pd.read_parquet(STATIC_ATTR_PARQUET)
need_cols = {"basin_id"}
if not need_cols.issubset(static.columns):
    raise ValueError("Static attributes must include 'basin_id'.")

# Reduce static to unique (in case of extras)
static = static.groupby("basin_id", as_index=False).first()

feat = feat.merge(static, on="basin_id", how="left")
print("Feature table shape (with statics):", feat.shape)

Feature table shape (with statics): (36924, 133)


In [17]:
feat

,basin_id,date_local,potential_evaporation_sum_1d,potential_evaporation_sum_3d,potential_evaporation_sum_7d,potential_evaporation_sum_14d,potential_evaporation_sum_30d,precipitation_sum_1d,precipitation_sum_3d,precipitation_sum_7d,...,dem_cv_y,slope_mean_y,slope_p90_deg_y,relief_m_y,log_acc_mean_y,log_acc_max_y,pct_slope_gt_a_y,pct_slope_gt_b_y,acc_mean_y,acc_max_y
0,3.0,1991-04-22,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,0.403294,23.063430,35.971821,6956.0,6.976817,13.794746,77.342031,25.226937,1070.502172,979449.0
1,3.0,1991-04-23,-0.000256,NaN,NaN,NaN,NaN,0.001312,NaN,NaN,...,0.403294,23.063430,35.971821,6956.0,6.976817,13.794746,77.342031,25.226937,1070.502172,979449.0
2,3.0,1991-04-24,-0.000298,NaN,NaN,NaN,NaN,0.002204,NaN,NaN,...,0.403294,23.063430,35.971821,6956.0,6.976817,13.794746,77.342031,25.226937,1070.502172,979449.0
3,3.0,1991-04-25,-0.000290,-0.000845,NaN,NaN,NaN,0.002312,0.005829,NaN,...,0.403294,23.063430,35.971821,6956.0,6.976817,13.794746,77.342031,25.226937,1070.502172,979449.0
4,3.0,1991-04-26,-0.000319,-0.000908,NaN,NaN,NaN,0.000747,0.005263,NaN,...,0.403294,23.063430,35.971821,6956.0,6.976817,13.794746,77.342031,25.226937,1070.502172,979449.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
36919,8.0,2024-12-27,-0.000212,-0.000729,-0.001557,-0.003308,-0.006572,0.000111,0.000580,0.002076,...,0.446350,23.648238,36.667324,6993.0,7.276396,14.060585,79.314210,26.663519,1444.768641,1277715.0
36920,8.0,2024-12-28,-0.000317,-0.000761,-0.001675,-0.003462,-0.006674,0.000000,0.000412,0.001725,...,0.446350,23.648238,36.667324,6993.0,7.276396,14.060585,79.314210,26.663519,1444.768641,1277715.0
36921,8.0,2024-12-29,-0.000310,-0.000839,-0.001778,-0.003584,-0.006738,0.000080,0.000191,0.001142,...,0.446350,23.648238,36.667324,6993.0,7.276396,14.060585,79.314210,26.663519,1444.768641,1277715.0
36922,8.0,2024-12-30,-0.000225,-0.000851,-0.001777,-0.003488,-0.006637,0.000880,0.000960,0.001778,...,0.446350,23.648238,36.667324,6993.0,7.276396,14.060585,79.314210,26.663519,1444.768641,1277715.0


In [ ]:
# Quick NA diagnostics
feature_cols = [c for c in feat.columns if c not in {"basin_id","date_local"}]
feat["feature_na_count"] = feat[feature_cols].isna().sum(axis=1)
feat["feature_na_frac"]  = feat["feature_na_count"] / len(feature_cols)

print("Avg NA frac:", round(feat["feature_na_frac"].mean(), 4))

Feature table shape (with statics): (36924, 108)
Avg NA frac: 0.0005


## Save Features-Only Table

In [8]:
# === Save engineered features (no target) ===
feat_sorted = feat.sort_values(["basin_id","date_local"]).reset_index(drop=True)
feat_sorted.to_parquet(FEATURES_DAILY_PARQUET, index=False)
print("Wrote features:", FEATURES_DAILY_PARQUET, "| rows:", len(feat_sorted), "| cols:", len(feat_sorted.columns))


Wrote features: /Users/liuq13/bhutan_climate_modeling/data/modeling/features/era5_features_basin_daily.parquet | rows: 36924 | cols: 110


## Build Train (Exogenous-Only)

In [9]:
# === Join features ↔ target (exogenous-only) ===
train_exog = (feat_sorted
              .merge(y[["basin_id","date_local","discharge_cms","qc_any"]],
                     on=["basin_id","date_local"], how="left")
              .sort_values(["basin_id","date_local"])
              .reset_index(drop=True))

# Basic QA
n_all = len(train_exog)
n_miss_target = train_exog["discharge_cms"].isna().sum()
print(f"Train exogenous rows: {n_all} | missing target rows: {n_miss_target}")

# Write
train_exog.to_parquet(TRAIN_EXOG_PARQUET, index=False)
print("Wrote exogenous train:", TRAIN_EXOG_PARQUET)


Train exogenous rows: 36924 | missing target rows: 7126
Wrote exogenous train: /Users/liuq13/bhutan_climate_modeling/data/modeling/datasets/train_basin_daily_exogenous.parquet


## Build Train (ARX: add lagged discharge)

In [10]:
if PRODUCE_ARX:
    te = train_exog.copy()

    # Add discharge lags & rolling stats per basin (leak-safe)
    te = te.sort_values(["basin_id","date_local"]).reset_index(drop=True)
    g = te.groupby("basin_id", group_keys=False)

    for L in ARX_LAGS:
        te[f"q_lag_{L}d"] = g["discharge_cms"].shift(L)

    for win, stat in ROLL_Q_STATS:
        if stat == "mean":
            te[f"q_roll{win}_mean"] = g["discharge_cms"].apply(lambda s: s.shift(1).rolling(win, min_periods=win).mean())
        elif stat == "std":
            te[f"q_roll{win}_std"]  = g["discharge_cms"].apply(lambda s: s.shift(1).rolling(win, min_periods=win).std())

    # Drop rows without full lag history or missing target
    needed = [f"q_lag_{L}d" for L in ARX_LAGS]
    before = len(te)
    te = te.dropna(subset=["discharge_cms"] + needed).reset_index(drop=True)
    after = len(te)
    print(f"ARX rows: {after} (dropped {before-after} due to lag history or missing target)")

    te.to_parquet(TRAIN_ARX_PARQUET, index=False)
    print("Wrote ARX train:", TRAIN_ARX_PARQUET)
else:
    print("PRODUCE_ARX=False → skipping ARX dataset.")


ARX rows: 29666 (dropped 7258 due to lag history or missing target)
Wrote ARX train: /Users/liuq13/bhutan_climate_modeling/data/modeling/datasets/train_basin_daily_arx.parquet
